# Water Quality Prediction: Benchmark Notebook 

## Challenge Overview

Welcome to the EY AI & Data Challenge 2026!  
The objective of this challenge is to build a robust **machine learning model** capable of predicting water quality across various river locations in South Africa. In addition to accurate predictions, the model should also identify and emphasize the key factors that significantly influence water quality.

Participants will be provided with a dataset containing three water quality parameters — **Total Alkalinity**, **Electrical Conductance**, and **Dissolved Reactive Phosphorus** — collected between 2011 and 2015 from approximately 200 river locations across South Africa. Each data point includes the geographic coordinates (latitude and longitude) of the sampling site, the date of collection, and the corresponding water quality measurements.

Using this dataset, participants are expected to build a machine learning model to predict water quality parameters for a separate validation dataset, which includes locations from different regions not present in the training data. The challenge also encourages participants to explore feature importance and provide insights into the factors most strongly associated with variations in water quality.

This challenge is designed for participants with varying levels of experience in data science, remote sensing, and environmental analytics. It offers a valuable opportunity to apply machine learning techniques to real-world environmental data and contribute to advancing water quality monitoring using artificial intelligence.


**About the Notebook:**  

In this notebook, we demonstrate a basic workflow that serves as a foundation for the challenge. The model has been developed to predict **water quality parameters** using features derived from the **Landsat** and **TerraClimate** datasets. Specifically, four spectral bands — **SWIR22** (Shortwave Infrared 2), **NIR** (Near Infrared), **Green**, and **SWIR16** (Shortwave Infrared 1) — were utilized from Landsat, along with derived spectral indices such as **NDMI** (Normalized Difference Moisture Index) and **MNDWI** (Modified Normalized Difference Water Index). In addition, the **PET** (Potential Evapotranspiration) variable was incorporated from the **TerraClimate** dataset to account for climatic influences on water quality.

The dataset spans a five-year period from **2011 to 2015**. Using **API-based data extraction** methods, both Landsat and TerraClimate features were retrieved directly from the [Microsoft Planetary Computer portal](https://planetarycomputer.microsoft.com).

These combined spectral, index-based, and climatic features were used as predictors in a regression model to estimate three key water quality parameters: **Total Alkalinity (TA)**, **Electrical Conductance (EC)**, and **Dissolved Reactive Phosphorus (DRP)**.

Please note that this notebook serves only as a starting point. Several assumptions were made during the data extraction and model development process, which you may find opportunities to improve upon. Participants are encouraged to explore additional features, enhance preprocessing techniques, or experiment with different regression algorithms to optimize predictive performance.


## Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process.

In [1]:
!pip install --user -r requirements.txt



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\phamt\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
try:
    import snowflake
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("Connected to Snowflake session.")
except Exception:
    session = None
    print("Running in local mode (no Snowflake session).")

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os 


Running in local mode (no Snowflake session).


## Response Variable

Before building the model, we first load the **water quality training dataset**. The curated dataset contains samples collected from various monitoring stations across the study region. Each record includes the geographical coordinates (Latitude and Longitude), the sample collection date, and the corresponding **measured values** for the three key water quality parameters — **Total Alkalinity (TA)**, **Electrical Conductance (EC)**, and **Dissolved Reactive Phosphorus (DRP)**.


In [3]:
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
display(Water_Quality_df.head(5))

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


## Predictor Variables

Now that we have our water quality dataset, the next step is to gather the predictor variables from the **Landsat** and **TerraClimate** datasets. In this notebook, we demonstrate how to **load previously extracted satellite and climate data** from separate files, rather than performing the extraction directly, which allows for a smoother and faster experience. Participants can refer to the dedicated extraction notebooks—one for Landsat and another for TerraClimate—to understand how the data was retrieved and processed, and they can also generate their own output CSV files if needed. Using these pre-extracted CSV files, this notebook focuses on loading the predictor features and running the subsequent analysis and model training efficiently.

For more detailed guidance on the original data extraction process, you can review the Landsat and TerraClimate example notebooks available on the Planetary Computer portal:

- [Landsat-c2-l2 - Example-Notebook](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2#Example-Notebook)  
- [Terraclimate - Example-Notebook](https://planetarycomputer.microsoft.com/dataset/terraclimate#Example-Notebook)

We have used selected spectral bands — **SWIR22** (Shortwave Infrared 2), **NIR** (Near Infrared), **Green**, and **SWIR16** (Shortwave Infrared 1) — and computed key spectral indices such as **NDMI** (Normalized Difference Moisture Index) and **MNDWI** (Modified Normalized Difference Water Index). These features capture surface moisture, vegetation, and water content characteristics that influence water quality variability.

In addition to Landsat features, we also incorporated the **Potential Evapotranspiration (PET)** variable from the **TerraClimate** dataset, which provides high-resolution global climate data. The PET feature captures the atmospheric demand for moisture, representing climatic conditions such as temperature, humidity, and radiation that influence surface water evaporation and thus affect water quality parameters.

The predictor features include:

- **SWIR22** – Sensitive to surface moisture and turbidity variations in water bodies.  
- **NIR** – Helps in identifying vegetation and suspended matter in water.  
- **Green** – Useful for detecting water color and surface reflectance changes.  
- **SWIR16** – Provides information on surface dryness and sediment concentration.  
- **NDMI** – Derived from NIR and SWIR16, indicates moisture and vegetation–water interaction.  
- **MNDWI** – Derived from Green and SWIR22, effective for distinguishing open water areas and reducing built-up noise.  
- **PET** – Extracted from the TerraClimate dataset, represents potential evapotranspiration influencing hydrological and water quality dynamics.


### **Tip 1**

Participants are encouraged to experiment with different combinations of **Landsat** bands or even include data from other public satellite data sources. By creating mathematical combinations of bands, you can derive various spectral indices that capture surface and environmental characteristics.


### Loading Pre-Extracted Landsat Data

In this notebook, we **load previously extracted Landsat data** from CSV files generated in a separate extraction notebook. This approach ensures a smoother and faster workflow, allowing participants to focus on data analysis and model development without waiting for time-consuming data retrieval.

Participants are expected to generate their own data extraction CSV files by running the dedicated Landsat extraction notebook. These CSV files can then be used here to smoothly run this benchmark notebook. Participants can refer to the extraction notebook to understand the API-based process, including how individual bands and indices like **NDMI** were computed. Using these pre-extracted CSV files simplifies preprocessing and is ideal for large-scale environmental and water quality analysis.


### **Tip 2**

In the data extraction process (performed in the dedicated extraction notebooks), a 100 m focal buffer was applied around each sampling location rather than using a single point. Participants may explore creating different focal buffers around the locations (e.g., 50 m, 150 m, etc.) during extraction. For example, if a 50 m buffer was used for “Band 2”, the extracted CSV values would reflect the average of Band 2 within 50 meters of each location. This approach can help reduce errors associated with spatial autocorrelation.


In [4]:
landsat_train_features = pd.read_csv("landsat_features_training.csv")
display(landsat_train_features.head(5))

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


In [5]:
# If NDMI and MNDWI columns are of type object, convert them to float
landsat_train_features['NDMI'] = landsat_train_features['NDMI'].astype(float)
landsat_train_features['MNDWI'] = landsat_train_features['MNDWI'].astype(float)


### Loading Pre-Extracted TerraClimate Data

In this notebook, we **load previously extracted TerraClimate data** from CSV files generated in a dedicated extraction notebook. This approach ensures a smoother and faster workflow, allowing participants to focus on data analysis and model development without waiting for time-consuming data retrieval.

Participants are expected to generate their own data extraction CSV files by running the dedicated TerraClimate extraction notebook. These CSV files can then be used here to smoothly run this benchmark notebook. Participants can refer to the extraction notebook to understand the API-based process, including how climate variables such as **Potential Evapotranspiration (PET)** were extracted. Using these pre-extracted CSV files ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis.


In [6]:
Terraclimate_df = pd.read_csv("terraclimate_features_training.csv")
display(Terraclimate_df.head(5))

,Latitude,Longitude,Sample Date,pet
0,-28.760833,17.730278,02-01-2011,174.2
1,-26.861111,28.884722,03-01-2011,124.1
2,-26.450000,28.085833,03-01-2011,127.5
3,-27.671111,27.236944,03-01-2011,129.7
4,-27.356667,27.286389,03-01-2011,129.2


## Joining the Predictor Variables and Response Variables

Now that we have extracted our predictor variables, we need to join them with the response variables. We use the **combine_two_datasets** function to merge the predictor variables and response variables. The **concat** function from pandas is particularly useful for this step.


In [7]:
# Combine two datasets vertically (along columns) using pandas concat function.
def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

In [8]:
# Combining ground data and final data into a single dataset.
wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
display(wq_data.head(5))

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,nir,green,swir16,swir22,NDMI,MNDWI,pet
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595,174.2
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,124.1
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,127.5
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,129.7
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,129.2


### Handling Missing Values

Before model training, missing values in the dataset were carefully handled to ensure data consistency and prevent model bias. Numerical columns were imputed using their median values, maintaining the overall data distribution while minimizing the impact of outliers.


In [9]:
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
wq_data.isna().sum()

Latitude                         0
Longitude                        0
Sample Date                      0
Total Alkalinity                 0
Electrical Conductance           0
Dissolved Reactive Phosphorus    0
nir                              0
green                            0
swir16                           0
swir22                           0
NDMI                             0
MNDWI                            0
pet                              0
dtype: int64

## Systematic Experiment Tracker (K-Fold + Checklist)

Muc tieu: theo doi R2 mot cach he thong theo tung lan nang cap mo hinh.

Checklist de xuat (chay tuan tu):
1. Baseline 4 features + RF
2. Baseline 4 features + XGBoost (hoac fallback GradientBoosting)
3. + Full Landsat features (nir, green, swir16)
4. + Spatial/Temporal features (Latitude, Longitude, month, year, season)
5. + Extra TerraClimate features (neu da trich xuat them: ppt, tmax, tmin, soil, q, ...)
6. Hyperparameter tuning cho tung target (TA, EC, DRP)

Moi test se duoc danh gia bang 5-fold CV va luu ket qua vao file r2_experiment_log.csv.

In [30]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor

# Optional dependency: XGBoost. Falls back automatically if not installed.
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

TARGET_COLS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

# Rebuild from source tables so experiments are independent from downstream baseline reductions.
exp_df = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df).copy()
exp_df['Sample Date'] = pd.to_datetime(exp_df['Sample Date'], dayfirst=True, errors='coerce')
exp_df['month'] = exp_df['Sample Date'].dt.month
exp_df['year'] = exp_df['Sample Date'].dt.year
exp_df['season'] = exp_df['month'].map({12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3})

# Candidate features currently available in this repo
base_4 = ['swir22', 'NDMI', 'MNDWI', 'pet']
landsat_plus = ['nir', 'green', 'swir16']
space_time = ['Latitude', 'Longitude', 'month', 'year', 'season']

# Optional TerraClimate extras if user has re-extracted and merged them
climate_extra_candidates = ['ppt', 'tmax', 'tmin', 'soil', 'q', 'aet', 'def', 'vpd', 'srad', 'ws']
climate_extras = [c for c in climate_extra_candidates if c in exp_df.columns]

feature_sets = {
    'T1_baseline4': base_4,
    'T2_baseline4_plus_xgb': base_4,
    'T3_landsat_plus': base_4 + [c for c in landsat_plus if c in exp_df.columns],
    'T4_space_time': base_4 + [c for c in landsat_plus if c in exp_df.columns] + [c for c in space_time if c in exp_df.columns],
    'T5_climate_extra': base_4 + [c for c in landsat_plus if c in exp_df.columns] + [c for c in space_time if c in exp_df.columns] + climate_extras,
}


def make_model(model_name: str):
    if model_name == 'rf':
        return RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
    if model_name == 'xgb':
        if HAS_XGB:
            return XGBRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=5,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1,
                objective='reg:squarederror'
            )
        return GradientBoostingRegressor(random_state=42)
    return GradientBoostingRegressor(random_state=42)


def evaluate_feature_set_cv(df, feature_cols, target_col, model_name='rf', n_splits=5):
    eval_df = df[feature_cols + [target_col]].dropna(subset=[target_col]).copy()

    X = eval_df[feature_cols]
    y = eval_df[target_col]

    pipe = Pipeline(steps=[
        ('imputer', KNNImputer(n_neighbors=5)),
        ('model', make_model(model_name))
    ])

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring={'r2': 'r2', 'rmse': 'neg_root_mean_squared_error'},
        n_jobs=-1,
        return_train_score=False
    )

    return {
        'R2_mean': float(np.mean(scores['test_r2'])),
        'R2_std': float(np.std(scores['test_r2'])),
        'RMSE_mean': float(-np.mean(scores['test_rmse'])),
        'RMSE_std': float(np.std(scores['test_rmse'])),
        'n_features': len(feature_cols)
    }


def run_experiment_checklist(df):
    tests = [
        ('T1_baseline4', 'rf'),
        ('T2_baseline4_plus_xgb', 'xgb'),
        ('T3_landsat_plus', 'xgb'),
        ('T4_space_time', 'xgb'),
        ('T5_climate_extra', 'xgb'),
    ]

    rows = []
    for test_name, model_name in tests:
        feat_cols = [c for c in feature_sets[test_name] if c in df.columns]
        if len(feat_cols) == 0:
            continue

        for target in TARGET_COLS:
            result = evaluate_feature_set_cv(df, feat_cols, target, model_name=model_name, n_splits=5)
            rows.append({
                'Test': test_name,
                'Model': model_name,
                'Target': target,
                'Features': ', '.join(feat_cols),
                **result
            })

    out = pd.DataFrame(rows)
    out = out.sort_values(['Target', 'R2_mean'], ascending=[True, False]).reset_index(drop=True)
    return out

experiment_results = run_experiment_checklist(exp_df)
experiment_results.to_csv('r2_experiment_log.csv', index=False)

print('Saved:', 'r2_experiment_log.csv')
print('XGBoost available:', HAS_XGB)
display(experiment_results)


Saved: r2_experiment_log.csv
XGBoost available: False


,Test,Model,Target,Features,R2_mean,R2_std,RMSE_mean,RMSE_std,n_features
0,T4_space_time,xgb,Dissolved Reactive Phosphorus,"swir22, NDMI, MNDWI, pet, nir, green, swir16, ...",0.560607,0.009362,33.779307,0.865463,12
1,T5_climate_extra,xgb,Dissolved Reactive Phosphorus,"swir22, NDMI, MNDWI, pet, nir, green, swir16, ...",0.560607,0.009362,33.779307,0.865463,12
2,T1_baseline4,rf,Dissolved Reactive Phosphorus,"swir22, NDMI, MNDWI, pet",0.544336,0.013373,34.390521,0.654611,4
3,T3_landsat_plus,xgb,Dissolved Reactive Phosphorus,"swir22, NDMI, MNDWI, pet, nir, green, swir16",0.278889,0.009199,43.271944,0.891032,7
4,T2_baseline4_plus_xgb,xgb,Dissolved Reactive Phosphorus,"swir22, NDMI, MNDWI, pet",0.270671,0.005718,43.516651,0.778571,4
5,T4_space_time,xgb,Electrical Conductance,"swir22, NDMI, MNDWI, pet, nir, green, swir16, ...",0.739955,0.012028,174.294906,5.043522,12
6,T5_climate_extra,xgb,Electrical Conductance,"swir22, NDMI, MNDWI, pet, nir, green, swir16, ...",0.739955,0.012028,174.294906,5.043522,12
7,T1_baseline4,rf,Electrical Conductance,"swir22, NDMI, MNDWI, pet",0.604441,0.015579,214.974231,5.443087,4
8,T2_baseline4_plus_xgb,xgb,Electrical Conductance,"swir22, NDMI, MNDWI, pet",0.350326,0.007935,275.554995,4.789236,4
9,T3_landsat_plus,xgb,Electrical Conductance,"swir22, NDMI, MNDWI, pet, nir, green, swir16",0.344457,0.010481,276.786308,4.596155,7


## Model Building

Now let us select the columns required for our model-building exercise. We will consider only **SWIR22**, **NDMI**, and **MNDWI** from the Landsat data, and **PET** from the TerraClimate dataset as our predictor variables. It does not make sense to use latitude and longitude as predictor variables, as they do not have any direct impact on predicting the water quality parameters.


In [11]:
# Retaining all available satellite and climate features along with target variables.
wq_data = wq_data[['nir','green','swir16','swir22','NDMI','MNDWI','pet', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

### **Tip 3**

We are developing individual models for each water quality parameter using a common set of features: **SWIR22**, **NDMI**, **MNDWI**, and **PET**. However, participants are encouraged to experiment with different feature combinations to build more robust machine learning models.


## Helper Functions

### Train and Test Split
We will now split the data into 70% training data and 30% test data. Scikit-learn (sklearn) is a robust library for machine learning in Python. The `model_selection` module in scikit-learn provides the `train_test_split` function, which can be used for this purpose.

### Feature Scaling
Before initiating model training, we may need to perform various data preprocessing steps. Here, we demonstrate scaling of the variables **SWIR22**, **NDMI**, **MNDWI**, and **PET** using StandardScaler.

Feature scaling is an essential preprocessing step for numerical features. Many machine learning algorithms—such as gradient descent methods, KNN, and linear or logistic regression—require scaling to achieve optimal performance. Scikit-learn provides several scaling utilities. In this notebook, we use **StandardScaler**, which transforms the data so that each feature has a mean of 0 and a standard deviation of 1.

### Model Training
Now that we have the data in a format suitable for machine learning, we can begin training our models. In this demonstration notebook, we build three separate regression models—one for each target water quality parameter: **Total Alkalinity**, **Electrical Conductance**, and **Dissolved Reactive Phosphorus**. Each model is trained independently to capture the unique relationships between the satellite-derived features and each water quality parameter.

We use the **Random Forest Regressor** from the scikit-learn library for model training. Scikit-learn offers a wide range of regression algorithms, along with powerful parameter tuning and customization options.

For model training, the predictor variables (e.g., SWIR22, NDMI, MNDWI, and PET) are stored in an array `X`, and the response variable (one of the water quality parameters) is stored in an array `Y`. Note that the response variable should not be included in `X`. Also, latitude, longitude, and sample date are excluded from the predictor variables since they serve only as spatial and temporal references.

### Model Evaluation
After training the models for the three water quality parameters, the next step is to evaluate their performance. Each regression model—Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus—is assessed using:

- **R² Score**: Measures how well the model explains the variance in the observed values.  
- **RMSE (Root Mean Square Error)**: Quantifies the average magnitude of prediction errors.

Together, these metrics help determine how effectively each model captures variations in water quality across locations and sampling dates. Scikit-learn provides built-in functions to compute both metrics. Participants may also explore additional evaluation techniques or custom metrics to enhance model assessment.


### **Tip 4**

There are many data preprocessing methods available that may help improve model performance. Participants are encouraged to explore various preprocessing techniques as well as different machine learning algorithms to build a more robust model.


In [12]:
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train):
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n{dataset_name} Evaluation:")
    print(f"R2: {r2:.3f}")
    print(f"RMSE: {rmse:.3f}")
    return y_pred, r2, rmse


## Model Workflow (Pipeline)

The complete model development process follows a structured pipeline to ensure consistency, reproducibility, and clarity. Each stage in the workflow is modularized into independent functions that can be reused for different water quality parameters. This modular approach streamlines the process and makes the workflow easily adaptable to new datasets or parameters in the future.

The pipeline automates the sequence of steps — from data preparation to evaluation — for each target parameter. The same set of predictor variables is used, while the response variable changes for each of the three targets: *Total Alkalinity (TA)*, *Electrical Conductance (EC)*, and *Dissolved Reactive Phosphorus (DRP)*. By maintaining a consistent framework, comparisons across models remain fair and interpretable.


In [13]:
def run_pipeline(X, y, param_name="Parameter"):
    print(f"\n{'='*60}")
    print(f"Training Model for {param_name}")
    print(f"{'='*60}")
    
    # Split data
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # Scale
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    
    # Train
    model = train_model(X_train_scaled, y_train)
    
    # Evaluate (in-sample)
    y_train_pred, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")
    
    # Evaluate (out-sample)
    y_test_pred, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, "Test")
    
    # Return summary
    results = {
        "Parameter": param_name,
        "R2_Train": r2_train,
        "RMSE_Train": rmse_train,
        "R2_Test": r2_test,
        "RMSE_Test": rmse_test
    }
    return model, scaler, pd.DataFrame([results])

### Model Training and Evaluation for Each Parameter

In this step, we apply the complete modeling pipeline to each of the three selected water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. The input feature set (`X`) remains the same across all three models, while the target variable (`y`) changes for each parameter. 

For every parameter, the `run_pipeline()` function is executed, which handles data preprocessing, model training, and both in-sample and out-of-sample evaluation. This ensures a consistent workflow and allows for a fair comparison of model performance across different water quality indicators.


In [14]:
X = wq_data.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])

y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

model_TA, scaler_TA, results_TA = run_pipeline(X, y_TA, "Total Alkalinity")
model_EC, scaler_EC, results_EC = run_pipeline(X, y_EC, "Electrical Conductance")
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")


Training Model for Total Alkalinity

Train Evaluation:
R2: 0.689
RMSE: 41.469

Test Evaluation:
R2: 0.450
RMSE: 55.968

Training Model for Electrical Conductance

Train Evaluation:
R2: 0.697
RMSE: 188.399

Test Evaluation:
R2: 0.492
RMSE: 243.470

Training Model for Dissolved Reactive Phosphorus

Train Evaluation:
R2: 0.639
RMSE: 30.535

Test Evaluation:
R2: 0.443
RMSE: 38.269


### Model Performance Summary

After training and evaluating the models for each water quality parameter, the individual performance metrics are combined into a single summary table. This table consolidates the R² and RMSE values for both in-sample and out-of-sample evaluations, enabling an easy comparison of model performance across Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. 

Such a summary provides a quick overview of how well each model captures the variability in each parameter and highlights any differences in predictive accuracy.


In [15]:
results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
results_summary

,Parameter,R2_Train,RMSE_Train,R2_Test,RMSE_Test
0,Total Alkalinity,0.688913,41.468883,0.450036,55.968272
1,Electrical Conductance,0.696563,188.398933,0.492285,243.470375
2,Dissolved Reactive Phosphorus,0.639388,30.535488,0.442896,38.268575


## Submission

Once you are satisfied with your model’s performance, you can proceed to make predictions for unseen data. To do this, use your trained model to estimate the concentrations of the target water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus — for a set of test locations provided in the **Submission_template.csv** file. 

The predicted results can then be uploaded to the challenge platform for evaluation.


In [16]:
test_file = pd.read_csv("submission_template.csv")
display(test_file.head(5))

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [17]:
landsat_val_features = pd.read_csv("landsat_features_validation.csv")
display(landsat_val_features.head(5))

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-32.043333,27.822778,01-09-2014,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052


In [18]:
Terraclimate_val_df = pd.read_csv("terraclimate_features_validation.csv")
display(Terraclimate_val_df.head(5))

,Latitude,Longitude,Sample Date,pet
0,-32.043333,27.822778,01-09-2014,161.90001
1,-33.329167,26.077500,16-09-2015,177.60000
2,-32.991639,27.640028,07-05-2015,158.40001
3,-34.096389,24.439167,07-02-2012,130.00000
4,-32.000556,28.581667,01-10-2014,152.50000


Similarly, participants can use the **Landsat** and **TerraClimate** data extraction demonstration notebooks to produce feature CSVs for their **validation** data. For convenience, we have already computed and saved example validation outputs as `landsat_features_val_V3.csv` and `Terraclimate_val_df_v3.csv`. 

Participants should save their own extracted files in the same format and column schema; doing so will allow this benchmark notebook to load the validation features directly and run smoothly.


In [19]:
#Consolidate all the extracted bands and features in a single dataframe
val_data = pd.DataFrame({
    'Longitude': landsat_val_features['Longitude'].values,
    'Latitude': landsat_val_features['Latitude'].values,
    'Sample Date': landsat_val_features['Sample Date'].values,
    'nir': landsat_val_features['nir'].values,
    'green': landsat_val_features['green'].values,
    'swir16': landsat_val_features['swir16'].values,
    'swir22': landsat_val_features['swir22'].values,
    'NDMI': landsat_val_features['NDMI'].values,
    'MNDWI': landsat_val_features['MNDWI'].values,
    'pet': Terraclimate_val_df['pet'].values,
})

In [20]:
# Impute the missing values
val_data = val_data.fillna(val_data.median(numeric_only=True))

In [21]:
# Extracting all feature columns from the validation dataset
submission_val_data=val_data.loc[:,['nir','green','swir16','swir22','NDMI','MNDWI','pet']]
display(submission_val_data.head())

,nir,green,swir16,swir22,NDMI,MNDWI,pet
0,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,161.90001
1,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,177.60000
2,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,158.40001
3,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,130.00000
4,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,152.50000


In [22]:
submission_val_data.shape

(200, 7)

In [23]:
# --- Predicting for Total Alkalinity ---
X_sub_scaled_TA = scaler_TA.transform(submission_val_data)
pred_TA_submission = model_TA.predict(X_sub_scaled_TA)

# --- Predicting for Electrical Conductance ---
X_sub_scaled_EC = scaler_EC.transform(submission_val_data)
pred_EC_submission = model_EC.predict(X_sub_scaled_EC)

# --- Predicting for Dissolved Reactive Phosphorus ---
X_sub_scaled_DRP = scaler_DRP.transform(submission_val_data)
pred_DRP_submission = model_DRP.predict(X_sub_scaled_DRP)

In [24]:
submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'].values,
    'Latitude': test_file['Latitude'].values,
    'Sample Date': test_file['Sample Date'].values,
    'Total Alkalinity': pred_TA_submission,
    'Electrical Conductance': pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission
})

In [25]:
#Displaying the sample submission dataframe
display(submission_df.head())

,Longitude,Latitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,27.822778,-32.043333,01-09-2014,118.336061,272.295835,45.717689
1,26.077500,-33.329167,16-09-2015,166.141759,623.883523,68.984807
2,27.640028,-32.991639,07-05-2015,66.501577,379.768372,36.875566
3,24.439167,-34.096389,07-02-2012,79.526746,302.889633,22.764069
4,28.581667,-32.000556,01-10-2014,95.485159,333.012414,30.640617


In [26]:
#Dumping the predictions into a csv file.
submission_df.to_csv("/tmp/submission.csv",index = False)

In [27]:
session.sql(f"""
    PUT file:///tmp/submission.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")

AttributeError: 'NoneType' object has no attribute 'sql'

### Upload submission file on platform

Upload the `submission.csv` file on the challenge platform to generate your score on the leaderboard.


## Conclusion

Now that you have learned a basic approach to model training, it’s time to explore your own techniques and ideas! Feel free to modify any of the functions presented in this notebook to experiment with alternative preprocessing steps, feature engineering strategies, or machine learning algorithms. 

We look forward to seeing your enhanced model and the insights you uncover. Best of luck with the challenge!


## Quick Run: Test 1 Only (Baseline 4 Features + RandomForest + 5-fold CV)

In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_validate
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Load source datasets
Water_Quality_df = pd.read_csv('water_quality_training_dataset.csv')
landsat_train_features = pd.read_csv('landsat_features_training.csv')
Terraclimate_df = pd.read_csv('terraclimate_features_training.csv')

# Same merge logic as baseline
exp_df = pd.concat([Water_Quality_df, landsat_train_features, Terraclimate_df], axis=1)
exp_df = exp_df.loc[:, ~exp_df.columns.duplicated()]

feature_cols = ['swir22', 'NDMI', 'MNDWI', 'pet']
target_cols = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

rows = []
for target_col in target_cols:
    eval_df = exp_df[feature_cols + [target_col]].dropna(subset=[target_col]).copy()
    X = eval_df[feature_cols]
    y = eval_df[target_col]

    pipe = Pipeline(steps=[
        ('imputer', KNNImputer(n_neighbors=5)),
        ('model', RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1))
    ])

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring={'r2': 'r2', 'rmse': 'neg_root_mean_squared_error'},
        n_jobs=-1,
        return_train_score=False
    )

    rows.append({
        'Test': 'T1_baseline4',
        'Model': 'rf',
        'Target': target_col,
        'Features': ', '.join(feature_cols),
        'R2_mean': float(np.mean(scores['test_r2'])),
        'R2_std': float(np.std(scores['test_r2'])),
        'RMSE_mean': float(-np.mean(scores['test_rmse'])),
        'RMSE_std': float(np.std(scores['test_rmse'])),
        'n_features': len(feature_cols)
    })

t1_results = pd.DataFrame(rows).sort_values('Target').reset_index(drop=True)
t1_results.to_csv('r2_experiment_log_test1.csv', index=False)

display(t1_results)
print('Saved: r2_experiment_log_test1.csv')

,Test,Model,Target,Features,R2_mean,R2_std,RMSE_mean,RMSE_std,n_features
0,T1_baseline4,rf,Dissolved Reactive Phosphorus,"swir22, NDMI, MNDWI, pet",0.544336,0.013373,34.390521,0.654611,4
1,T1_baseline4,rf,Electrical Conductance,"swir22, NDMI, MNDWI, pet",0.604441,0.015579,214.974231,5.443087,4
2,T1_baseline4,rf,Total Alkalinity,"swir22, NDMI, MNDWI, pet",0.556382,0.012525,49.725928,0.573644,4


Saved: r2_experiment_log_test1.csv


In [31]:
# Compare all tests against Test 1 baseline
all_results = experiment_results.copy()
all_results = all_results[['Test', 'Model', 'Target', 'R2_mean', 'R2_std', 'RMSE_mean', 'RMSE_std', 'n_features']]

baseline = all_results[all_results['Test'] == 'T1_baseline4'][['Target', 'R2_mean']].rename(columns={'R2_mean': 'R2_baseline'})
comparison = all_results.merge(baseline, on='Target', how='left')
comparison['Delta_R2_vs_T1'] = comparison['R2_mean'] - comparison['R2_baseline']
comparison = comparison.sort_values(['Target', 'R2_mean'], ascending=[True, False]).reset_index(drop=True)

comparison.to_csv('r2_experiment_comparison.csv', index=False)
display(comparison)
print('Saved: r2_experiment_comparison.csv')

,Test,Model,Target,R2_mean,R2_std,RMSE_mean,RMSE_std,n_features,R2_baseline,Delta_R2_vs_T1
0,T4_space_time,xgb,Dissolved Reactive Phosphorus,0.560607,0.009362,33.779307,0.865463,12,0.544336,0.016271
1,T5_climate_extra,xgb,Dissolved Reactive Phosphorus,0.560607,0.009362,33.779307,0.865463,12,0.544336,0.016271
2,T1_baseline4,rf,Dissolved Reactive Phosphorus,0.544336,0.013373,34.390521,0.654611,4,0.544336,0.000000
3,T3_landsat_plus,xgb,Dissolved Reactive Phosphorus,0.278889,0.009199,43.271944,0.891032,7,0.544336,-0.265447
4,T2_baseline4_plus_xgb,xgb,Dissolved Reactive Phosphorus,0.270671,0.005718,43.516651,0.778571,4,0.544336,-0.273666
5,T4_space_time,xgb,Electrical Conductance,0.739955,0.012028,174.294906,5.043522,12,0.604441,0.135514
6,T5_climate_extra,xgb,Electrical Conductance,0.739955,0.012028,174.294906,5.043522,12,0.604441,0.135514
7,T1_baseline4,rf,Electrical Conductance,0.604441,0.015579,214.974231,5.443087,4,0.604441,0.000000
8,T2_baseline4_plus_xgb,xgb,Electrical Conductance,0.350326,0.007935,275.554995,4.789236,4,0.604441,-0.254115
9,T3_landsat_plus,xgb,Electrical Conductance,0.344457,0.010481,276.786308,4.596155,7,0.604441,-0.259984


Saved: r2_experiment_comparison.csv


## Heavy Test: Target-Specific Hyperparameter Tuning (Feasible Now)

This run focuses on what can be tested immediately without re-extracting remote datasets:
- Feature engineering from existing columns (`swir_ratio`, `nir_green_ratio`)
- Spatial and temporal features
- KNN imputation
- Outlier trimming by target (1% tails)
- RandomizedSearchCV per target (RandomForest)

Outputs:
- `r2_tuning_results.csv`
- `r2_tuning_vs_best_previous.csv`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor

# Build feature table from existing files
wq = pd.read_csv('water_quality_training_dataset.csv')
ls = pd.read_csv('landsat_features_training.csv')
tc = pd.read_csv('terraclimate_features_training.csv')

df = pd.concat([wq, ls, tc], axis=1)
df = df.loc[:, ~df.columns.duplicated()].copy()

df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')
df['month'] = df['Sample Date'].dt.month
df['year'] = df['Sample Date'].dt.year
df['season'] = df['month'].map({12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3})

# New engineered features from existing columns
eps = 1e-10
df['swir_ratio'] = df['swir22'] / (df['swir16'] + eps)
df['nir_green_ratio'] = df['nir'] / (df['green'] + eps)

feature_cols = [
    'nir','green','swir16','swir22','NDMI','MNDWI','pet',
    'Latitude','Longitude','month','year','season',
    'swir_ratio','nir_green_ratio'
]

target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# Load previous best to compare
prev = pd.read_csv('r2_experiment_comparison.csv')
prev_best = prev.groupby('Target', as_index=False)['R2_mean'].max().rename(columns={'R2_mean': 'Prev_Best_R2'})

cv = KFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    'model__n_estimators': [200, 400, 600, 800, 1000],
    'model__max_depth': [None, 8, 12, 16, 24],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__min_samples_leaf': [1, 2, 4, 8],
    'model__max_features': ['sqrt', 'log2', 0.6, 0.8, 1.0]
}

rows = []
for target in target_cols:
    sub = df[feature_cols + [target]].dropna(subset=[target]).copy()

    # Outlier trimming on target only (1% each tail)
    low = sub[target].quantile(0.01)
    high = sub[target].quantile(0.99)
    sub = sub[(sub[target] >= low) & (sub[target] <= high)].copy()

    X = sub[feature_cols]
    y = sub[target]

    pipe = Pipeline([
        ('imputer', KNNImputer(n_neighbors=5)),
        ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
    ])

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=20,
        scoring='r2',
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    search.fit(X, y)

    # Re-evaluate best estimator with CV for stable report
    best_scores = cross_validate(
        search.best_estimator_,
        X,
        y,
        cv=cv,
        scoring={'r2': 'r2', 'rmse': 'neg_root_mean_squared_error'},
        n_jobs=-1,
        return_train_score=False
    )

    rows.append({
        'Target': target,
        'Model': 'T6_tuned_rf',
        'R2_mean': float(np.mean(best_scores['test_r2'])),
        'R2_std': float(np.std(best_scores['test_r2'])),
        'RMSE_mean': float(-np.mean(best_scores['test_rmse'])),
        'RMSE_std': float(np.std(best_scores['test_rmse'])),
        'n_features': len(feature_cols),
        'rows_used_after_trim': int(len(sub)),
        'best_params': str(search.best_params_)
    })


tuning_results = pd.DataFrame(rows)
tuning_vs_prev = tuning_results.merge(prev_best, on='Target', how='left')
tuning_vs_prev['Delta_vs_prev_best'] = tuning_vs_prev['R2_mean'] - tuning_vs_prev['Prev_Best_R2']


tuning_results.to_csv('r2_tuning_results.csv', index=False)
tuning_vs_prev.to_csv('r2_tuning_vs_best_previous.csv', index=False)

print('Saved: r2_tuning_results.csv')
print('Saved: r2_tuning_vs_best_previous.csv')
display(tuning_vs_prev.sort_values('Target').reset_index(drop=True))